# 05 - Evaluation (Phases 10 and 11) - run in GOOGLE COLAB (GPU)

**Part A (Phase 10):** official internal test set - accuracy, precision, recall, F1, macro-F1, confusion matrix, language-wise table, comparison with the TF-IDF baseline, zero-shot Telugu/Kannada.
**Part B (Phase 11):** IndicGuard external test - **the model is locked: no training, no tuning, no threshold change.**

Same setup as notebook 04 (GPU runtime, Drive folder with the data and the trained model `xlm_roberta_hate/`). Run top to bottom.
Nothing here is pre-filled: **every number is computed from your trained model.**

## 1. Bootstrap

In [ ]:
# ===================== Colab bootstrap (identical cell in notebooks 04 and 05) =====================
import os, sys, subprocess, getpass, shutil

GITHUB_USER = "YOUR-GITHUB-USERNAME"                 # <-- EDIT: your GitHub user name
REPO_NAME   = "multilingual-hate-speech"
DRIVE_ROOT  = "/content/drive/MyDrive/hate-speech"   # the folder you created in Google Drive

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = f"/content/{REPO_NAME}"
if not os.path.isdir(REPO_DIR):
    token = getpass.getpass("GitHub token (what you type is hidden): ")
    url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    r = subprocess.run(["git", "clone", url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed:\n" + r.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin",
                    f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"], check=True)   # do not keep the token
    print("cloned", REPO_NAME)
else:
    print("repo already cloned (to get newer code: Runtime > Disconnect and delete runtime, then run again)")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

# copy the data files from Google Drive into the repo folders
FILES = {"train.csv": "data/processed", "val.csv": "data/processed", "test.csv": "data/processed",
         "zero_shot_test.csv": "data/processed", "indicguard_external.csv": "data/external_test"}
for name, dest in FILES.items():
    src = f"{DRIVE_ROOT}/data/{name}"
    if os.path.exists(src):
        os.makedirs(dest, exist_ok=True); shutil.copy(src, f"{dest}/{name}"); print("copied", name)
    else:
        print("!! MISSING in Google Drive:", src)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Runtime > Change runtime type > T4 GPU > Save, then run again from the top."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Load the trained model (inference only) and lock it with a fingerprint

In [ ]:
import json, numpy as np, pandas as pd
from IPython.display import display, Image
from src.model.evaluate import (load_model, model_fingerprint, predict_probs, evaluate_frame, language_table, strict_external,
                                comparison_rows, baseline_vs_xlmr, plot_internal_vs_external, save_json)
from src.utils.metrics import plot_confusion_matrix
from src.data.io_utils import read_processed_csv
from src.utils.config import INTERNAL_LANGUAGES, EXTERNAL_LANGUAGES, FIGURES_DIR, METRICS_DIR
MODEL_DIR = f"{DRIVE_ROOT}/xlm_roberta_hate"
os.makedirs(FIGURES_DIR, exist_ok=True); os.makedirs(METRICS_DIR, exist_ok=True)

fingerprint = model_fingerprint(MODEL_DIR)
print("model SHA-256:", fingerprint)
tokenizer, model, device = load_model(MODEL_DIR)
train_cfg = json.load(open(f"{MODEL_DIR}/training_config.json"))
print("device:", device, "| trained for best epoch", train_cfg["best_epoch"], "| max_length", train_cfg["max_length"])
MAXLEN = train_cfg["max_length"]

# PART A - Internal test set (Phase 10)
## 3. Predict on the official test set

In [ ]:
test = read_processed_csv("data/processed/test.csv")
test["text"] = test["text"].fillna("")
probs_test = predict_probs(model, tokenizer, test["text"].tolist(), max_length=MAXLEN)
res_test, pred_test = evaluate_frame(test, probs_test, group_cols=("language", "source"))
o = res_test["overall"]
print(f"TEST (n={o['n']:,})  accuracy {o['accuracy']:.4f} | precision(hate) {o['precision_hate']:.4f} | recall(hate) {o['recall_hate']:.4f} | "
      f"F1(hate) {o['f1_hate']:.4f} | MACRO-F1 {o['macro_f1']:.4f} | ROC-AUC {o['roc_auc']:.4f}")

## 4. Confusion matrix -> `results/figures/confusion_matrix.png`

In [ ]:
plot_confusion_matrix(test["label"], pred_test, FIGURES_DIR / "confusion_matrix.png", "XLM-R - internal test")
display(Image(str(FIGURES_DIR / "confusion_matrix.png")))
print("tn, fp, fn, tp =", [res_test["overall"]["confusion_matrix"][k] for k in ("tn", "fp", "fn", "tp")])

## 5. Language-wise evaluation (English, Hindi, Tamil, Malayalam)
Precision / Recall / F1 are for the HATE class.

In [ ]:
lang_tab = language_table(res_test["by_language"], INTERNAL_LANGUAGES)
display(lang_tab.round(4))
lang_tab.to_csv(METRICS_DIR / "xlmr_test_by_language.csv", index=False)
print("\nBy source:"); display(pd.DataFrame(res_test["by_source"]).T[["n", "accuracy", "precision_hate", "recall_hate", "f1_hate", "macro_f1"]].round(4))

## 6. Comparison with the TF-IDF + Logistic Regression baseline

In [ ]:
bpath = METRICS_DIR / "baseline_metrics.json"
if bpath.exists():
    cmp = baseline_vs_xlmr(json.load(open(bpath)), res_test["by_language"])
    cmp.loc[len(cmp)] = ["ALL", json.load(open(bpath))["test"]["overall"]["macro_f1"], o["macro_f1"],
                         o["macro_f1"] - json.load(open(bpath))["test"]["overall"]["macro_f1"],
                         json.load(open(bpath))["test"]["overall"]["f1_hate"], o["f1_hate"]]
    display(cmp.round(4)); cmp.to_csv(METRICS_DIR / "model_comparison.csv", index=False)
else:
    print("baseline_metrics.json not found in the repo (commit results/metrics from VS Code first).")

## 7. Zero-shot check: Telugu and Kannada (languages the model never saw in training)

In [ ]:
zero = read_processed_csv("data/processed/zero_shot_test.csv"); zero["text"] = zero["text"].fillna("")
probs_zero = predict_probs(model, tokenizer, zero["text"].tolist(), max_length=MAXLEN)
res_zero, pred_zero = evaluate_frame(zero, probs_zero, group_cols=("language",))
display(pd.DataFrame(res_zero["by_language"]).T[["n", "accuracy", "precision_hate", "recall_hate", "f1_hate", "macro_f1", "roc_auc"]].round(4))
print("overall zero-shot macro-F1: %.4f  (TF-IDF baseline was 0.43)" % res_zero["overall"]["macro_f1"])

## 8. Save the internal results

In [ ]:
fp_internal = model_fingerprint(MODEL_DIR)
save_json({"model_sha256": fp_internal, "training_config": train_cfg, "test": res_test, "zero_shot": res_zero}, METRICS_DIR / "xlmr_test_metrics.json")
os.makedirs(f"{DRIVE_ROOT}/predictions", exist_ok=True)       # predictions contain real text -> Google Drive only, never GitHub
test.assign(p_hate=probs_test[:, 1], pred=pred_test).to_csv(f"{DRIVE_ROOT}/predictions/xlmr_test_predictions.csv", index=False)
print("saved results/metrics/xlmr_test_metrics.json and xlmr_test_by_language.csv")

# PART B - IndicGuard external test (Phase 11)

```
MACD + HateXplain -> TRAIN XLM-R -> LOCK MODEL -> IndicGuard -> TEST ONLY
```
Rules followed below: the model is loaded from disk and frozen; **no** fine-tuning; **no** threshold tuning (fixed 0.5);
the model fingerprint is compared before and after.

Two versions of IndicGuard are scored: **broad** (every row) and **strict** (jailbreak/persona prompts removed).
Read the numbers with these facts in mind (also write them in the report):
* IndicGuard is an LLM-safety dataset of *prompts*, machine-translated - not social-media comments.
* The 11 languages contain the **same prompts translated**, so the per-language results are not independent experiments.
* Hate is rare there (about 15%), so precision and F1 for the hate class are naturally lower than on the balanced internal test. **Compare ROC-AUC and PR-AUC too.**

In [ ]:
fp_before = model_fingerprint(MODEL_DIR)
ext_all = read_processed_csv("data/external_test/indicguard_external.csv")
ext_all["text"] = ext_all["text"].fillna("")
ext_all = ext_all[ext_all["language"].isin(EXTERNAL_LANGUAGES)].reset_index(drop=True)
ext_versions = {"IndicGuard (broad)": ext_all, "IndicGuard (strict)": strict_external(ext_all)}
for k, v in ext_versions.items():
    print(f"{k:<22} {len(v):>8,} prompts | hate share {100 * v['label'].mean():.1f}% | " + ", ".join(f"{l} {int((v.language == l).sum()):,}" for l in EXTERNAL_LANGUAGES))

## 9. Run the locked model on IndicGuard (a few minutes)

In [ ]:
ext_results, rows = {}, comparison_rows("Internal", res_test["by_language"], INTERNAL_LANGUAGES)
for name, d in ext_versions.items():
    probs = predict_probs(model, tokenizer, d["text"].tolist(), max_length=MAXLEN)
    r, pred = evaluate_frame(d, probs, group_cols=("language",))
    ext_results[name] = r
    rows += comparison_rows(name, r["by_language"], EXTERNAL_LANGUAGES)
    o2 = r["overall"]
    print(f"{name}: accuracy {o2['accuracy']:.4f} | precision(hate) {o2['precision_hate']:.4f} | recall(hate) {o2['recall_hate']:.4f} | "
          f"macro-F1 {o2['macro_f1']:.4f} | ROC-AUC {o2['roc_auc']:.4f} | PR-AUC {o2['pr_auc']:.4f}")
    if name.endswith("(broad)"):
        plot_confusion_matrix(d["label"], pred, FIGURES_DIR / "confusion_matrix_indicguard.png", "XLM-R - IndicGuard (broad)")

## 10. Internal test vs IndicGuard external test

In [ ]:
comp = pd.DataFrame(rows)
display(comp.round(4))
comp.to_csv(METRICS_DIR / "xlmr_internal_vs_external.csv", index=False)
plot_internal_vs_external(comp, FIGURES_DIR / "internal_vs_external_macro_f1.png", "Macro-F1")
display(Image(str(FIGURES_DIR / "internal_vs_external_macro_f1.png")))

## 11. Lock check - the model must be byte-for-byte identical before and after

In [ ]:
fp_after = model_fingerprint(MODEL_DIR)
assert fp_internal == fp_before == fp_after, "MODEL CHANGED - the external evaluation is NOT valid!"
print("OK - model unchanged during the whole evaluation:", fp_after[:16], "...")
save_json({"model_sha256_before": fp_before, "model_sha256_after": fp_after, "threshold": 0.5,
           "languages": EXTERNAL_LANGUAGES, "results": ext_results}, METRICS_DIR / "xlmr_external_metrics.json")

## 12. Export the results (numbers and figures only, no text) and download them

In [ ]:
import glob, zipfile
zpath = "/content/results_export.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob("results/metrics/xlmr_*") + glob.glob("results/metrics/model_comparison.csv") + glob.glob("results/figures/*.png"):
        z.write(p, p); print("added", p)
os.makedirs(f"{DRIVE_ROOT}/results_export", exist_ok=True); shutil.copy(zpath, f"{DRIVE_ROOT}/results_export/")
from google.colab import files
files.download(zpath)